# Fine-Tuning Laya on `train.csv` (Kaggle GPU T4 ×2 DDP)

This notebook adapts the Laya fine-tuning process for your 2-column dataset (`resp_text`, `clarity`). It maps your text to Laya's state input and sets up a 3-way classification decision dynamically.

---

### ⚠️ Kaggle Notebook Settings:
In the right-hand sidebar panel under **Notebook options**:
* **Accelerator:** Select **`GPU T4 x2`** (ensure both GPUs are allocated)
* **Internet:** Set to **`On`**
* **Output:** Checkpoints and final model are saved to `/kaggle/working/laya_finetuned_custom`


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [12]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


Tue Sep 29 22:50:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies


In [13]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


Laya version        : 0.3.22
Transformers version: 5.17.0
PyTorch version     : 2.10.0+cu128


## 3. Download & Preprocess Data for DDP
We read `train.csv` and map `resp_text` and `clarity` into Laya's expected JSON format on the fly.


In [14]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID, max_workers=1, etag_timeout=60)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("Loading custom dataset...")
# If using a Kaggle dataset path, update this (e.g., "/kaggle/input/datasets/juliaduboisas/train-csv/train.csv")
CSV_PATH = "/kaggle/input/datasets/juliaduboisas/train-csv/train.csv"
full_ds = load_dataset("csv", data_files=CSV_PATH, split="train")
splits = full_ds.train_test_split(test_size=0.1, seed=42)
ds_train = splits["train"]
ds_test = splits["test"]
ds_test.save_to_disk("/kaggle/working/test_split")
print(f"Created 90/10 split: {len(ds_train)} training rows, {len(ds_test)} reserved for testing.")

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    keys = list(crit.keys())
    target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[t],
        "target": target,
        "label": label
    }

items = []
for row in ds_train:
    # 1. Map `resp_text` into Laya's state context
    state = {"text": str(row["resp_text"])}
    
    # 2. Define the decision task natively
    questions = {
        "clarity": {
            "type": "choice",
            "instructions": "Avalie a clareza da seguinte resposta.",
            "criteria": {
                "c1": "Nível c1",
                "c234": "Nível c234",
                "c5": "Nível c5"
            }
        }
    }
    
    # 3. Map the true `clarity` string to proper one-hot probabilities
    label_str = str(row["clarity"])
    gold = {
        "clarity": {
            "label": label_str,
            "probabilities": {
                "c1": 1.0 if label_str == "c1" else 0.0,
                "c234": 1.0 if label_str == "c234" else 0.0,
                "c5": 1.0 if label_str == "c5" else 0.0
            }
        }
    }
    
    # 4. Generate token sequences
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it:
                items.append(it)

print(f"Preprocessed {len(items)} training sequences across {len(ds_train)} cases.")
torch.save(items, "/kaggle/working/train_items.pt")
print("Saved preprocessed items to /kaggle/working/train_items.pt")


Fetching tokenizer and config from convaiinnovations/laya...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

Loading custom dataset...


Saving the dataset (0/1 shards):   0%|          | 0/2010 [00:00<?, ? examples/s]

Created 90/10 split: 18082 training rows, 2010 reserved for testing.
Preprocessed 18082 training sequences across 18082 cases.
Saved preprocessed items to /kaggle/working/train_items.pt


## 4. DDP Training Script (`train_ddp.py`)


In [15]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    my_items = train_items[rank::world_size]
    
    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 4
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()
        
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-custom-finetune"
        cfg["temperature"] = fitted_temps
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_ddp.py


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`


In [16]:
OUTPUT_DIR = "/kaggle/working/laya_finetuned_custom"
MODEL_DIR = model_dir

cmd = f"torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print("Executing DDP training:", cmd)
!{cmd}


Executing DDP training: torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py /root/.cache/huggingface/hub/models--convaiinnovations--laya/snapshots/55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851 /kaggle/working/laya_finetuned_custom
W0929 22:50:42.506000 251 torch/distributed/run.py:852] 
W0929 22:50:42.506000 251 torch/distributed/run.py:852] *****************************************
W0929 22:50:42.506000 251 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0929 22:50:42.506000 251 torch/distributed/run.py:852] *****************************************
[W929 22:50:42.693611472 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W929 22:50:50.684576310 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W929 22:50:50.77

## 6. Run Benchmark Evaluation on Test Split
Evaluate the fine-tuned model against the reserved 10% test split, dynamically converting rows to states exactly as we did in preprocessing.


In [17]:
import time, json
import numpy as np
import pandas as pd
from datasets import load_from_disk
import laya
from laya.common import ece_score

print("Loading test split for evaluation...")
ds_test = load_from_disk("/kaggle/working/test_split")

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(ds_test)} test cases on GPU...")
t0_eval = time.time()

for i, row in enumerate(ds_test):
    case_id = f"case_{i}"
    workflow = "clarity_evaluation"
    
    state = {"text": str(row["resp_text"])}
    questions = {
        "clarity": {
            "type": "choice",
            "instructions": "Avalie a clareza da seguinte resposta.",
            "criteria": {
                "c1": "Nível c1",
                "c234": "Nível c234",
                "c5": "Nível c5"
            }
        }
    }
    
    label_str = str(row["clarity"])
    gold = {
        "clarity": {
            "label": label_str,
            "probabilities": {
                "c1": 1.0 if label_str == "c1" else 0.0,
                "c234": 1.0 if label_str == "c234" else 0.0,
                "c5": 1.0 if label_str == "c5" else 0.0
            }
        }
    }
    
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")


Loading test split for evaluation...
Evaluating 2010 test cases on GPU...
Evaluated all 2010 cases in 89.2s!


## 7. Compute Official Metrics


In [18]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        
        keys = list(qdef["criteria"].keys())
        pred_choice = p_ans["choice"]
        gold_label = str(g_ans["label"])
        
        is_corr = float(pred_choice == gold_label)
        accuracies.append(is_corr)
        all_corrects.append(is_corr)
        
        p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
        g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
        p_probs /= p_probs.sum()
        g_probs /= g_probs.sum()
        
        all_confs.append(float(p_probs.max()))
        soft_accuracies.append(float((p_probs * g_probs).sum()))
        brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
        tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
        kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "Laya (Custom Fine-Tune 2xT4)",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "ms/case": round(laya_latency, 1)
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


=== BENCHMARK TABLE ===

| Model                        |   Accuracy |   Soft Acc |   Brier |   ECE |   ms/case |
|:-----------------------------|-----------:|-----------:|--------:|------:|----------:|
| Laya (Custom Fine-Tune 2xT4) |      0.447 |      0.356 |   0.638 | 0.034 |      36.8 |


## 8. Save Benchmark Report as JSON


In [19]:
import json, os

report = {
    "benchmark": "train.csv",
    "model": "Laya (Custom Fine-Tune 2xT4)",
    "n_cases": len(ds_test),
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "latency_p50_ms": round(float(laya_latency), 1),
        "kl_divergence": round(float(laya_kl), 4)
    }
}

out_file1 = "/kaggle/working/laya_custom_benchmark_report.json"
out_file2 = os.path.join(OUTPUT_DIR, "benchmark_report.json")

with open(out_file1, "w") as f:
    json.dump(report, f, indent=2)

with open(out_file2, "w") as f:
    json.dump(report, f, indent=2)

print(f"Benchmark results successfully saved to: {out_file1}")


Benchmark results successfully saved to: /kaggle/working/laya_custom_benchmark_report.json
